# Predict labels from autoencoder embeddings

Uses the **encoder half of a trained autoencoder** (saved by
`autoencoder.ipynb` to its `CKPT_PATH`) as a frozen feature extractor, and
trains a small MLP head on the latent vector `z` to predict one plot label:

- `features/<plot_uid>.npz` — point cloud + extra dims (same `EXTRA_DIMS` as
  the autoencoder was trained with)
- `labels/<plot_uid>.json` — `{"labels": {KEY: value, ...}}` from prepare.py
- `CKPT_PATH` — the checkpoint written by `autoencoder.ipynb`

The encoder is **frozen** (`requires_grad_(False)`); only the head trains, so
this measures how much label information the embeddings carry. Plots are
embedded in chunks of `EMBED_BATCH` (the point-MLP intermediate is
`(B, 1024, N_POINTS)`, so one giant forward over all plots can OOM a GPU).
Pick `LABEL_KEY`, the usual training knobs, optional `PATIENCE`/`MIN_DELTA`
early stopping on the validation split, and the last cell reports RMSE/MAE on
the validation split with a true-vs-predicted scatter plus an embedding-norm
histogram.

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plots).

In [ ]:
import glob
import json
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
LABELS_DIR = "../dl_dataset/labels"      # prepare.py -o <out>/labels
LABEL_KEY = "VOL_TREES"                  # which label to predict (single head)
EXTRA_DIMS = ["semantic_code"]           # MUST match the autoencoder's EXTRA_DIMS
CKPT_PATH = "../checkpoints/autoencoder.pt"  # checkpoint written by autoencoder.ipynb
N_POINTS = 1024     # point count per cloud (the encoder is N-agnostic; keep the
                    # same value as autoencoder training for consistency)
N_SAMPLES = "all"   # how many plots to load: an int, or "all"
EMBED_BATCH = 32    # plots per encoder forward when embedding. The point-MLP
                    # intermediate is (B, 1024, N_POINTS) - embedding ALL plots
                    # in one forward can allocate tens of GB on a GPU
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
DROPOUT = 0.3    # dropout probability in the head (0 = off)
VAL_FRAC = 0.2
VAL_FEATURES_DIR = None  # optional dedicated validation features dir (a prepare.py output dir);
                     # None = random VAL_FRAC split of FEATURES_DIR
VAL_LABELS_DIR = None    # optional dedicated validation labels dir (required with VAL_FEATURES_DIR in label notebooks)
SEED = 42
STANDARDIZE_LABELS = True  # z-score the label using train-set stats
PATIENCE = None     # early stopping on the val loss: stop after this many
                    # epochs without improvement; None = off (needs a val split)
MIN_DELTA = 0.0     # minimum val-loss improvement that resets the patience counter

_first_feat = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.npz")))
_first_lab = sorted(glob.glob(os.path.join(LABELS_DIR, "*.json")))
if not _first_feat or not _first_lab:
    raise FileNotFoundError("run prepare.py first - no feature/label files found")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"no checkpoint at {CKPT_PATH!r} - run autoencoder.ipynb "
                            f"first (its CKPT_PATH)")
with np.load(_first_feat[0]) as z:
    print("available feature arrays:", z.files)
with open(_first_lab[0]) as fh:
    print("available labels:", sorted(json.load(fh)["labels"].keys()))

In [ ]:
class PlotDataset(Dataset):
    """(feature, label) pairs per plot: features/<uid>.npz + labels/<uid>.json."""

    def __init__(self, features_dir, labels_dir, label_key, extra_dims, n_points,
                 n_samples=None):
        self.n_points = n_points
        self.extra_dims = list(extra_dims)
        feats = sorted(glob.glob(os.path.join(features_dir, "*.npz")))
        if n_samples is not None:
            feats = feats[:n_samples]
        if not feats:
            raise FileNotFoundError(f"no feature files in {features_dir!r}")
        self.feats = feats
        self.names = [os.path.splitext(os.path.basename(f))[0] for f in feats]
        self.labels = []
        for name in self.names:
            with open(os.path.join(labels_dir, name + ".json")) as fh:
                lab = json.load(fh)["labels"]
            if label_key not in lab:
                raise KeyError(f"label {label_key!r} missing for plot {name!r}; "
                               f"available: {sorted(lab)}")
            self.labels.append(float(lab[label_key]))
        with np.load(feats[0]) as z:
            available = set(z.files)
        missing = [k for k in self.extra_dims if k not in available]
        if missing:
            raise KeyError(f"feature dims {missing} not in the .npz; "
                           f"available: {sorted(available)}")

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, i):
        with np.load(self.feats[i]) as z:
            cols = [z["points"].astype(np.float32)]          # (N, 3)
            for k in self.extra_dims:
                cols.append(z[k].astype(np.float32)[:, None])  # (N, 1)
        mat = np.concatenate(cols, axis=1)                   # (N, C)
        # fix the point count: tile if short, truncate if long
        n = len(mat)
        if n >= self.n_points:
            mat = mat[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            mat = np.tile(mat, (reps, 1))[: self.n_points]
        return torch.from_numpy(mat.T.copy()), self.labels[i]  # (C, N), scalar


dataset = PlotDataset(FEATURES_DIR, LABELS_DIR, LABEL_KEY, EXTRA_DIMS, N_POINTS,
                      n_samples=None if N_SAMPLES == "all" else N_SAMPLES)
ys = np.array(dataset.labels, dtype=np.float64)
print(f"loaded {len(dataset)} plot(s), label = {LABEL_KEY}")
print(f"label stats: n={len(ys)}  mean={ys.mean():.4g}  std={ys.std():.4g}  "
      f"min={ys.min():.4g}  max={ys.max():.4g}")

In [ ]:
def chamfer_loss(a, b):
    """Symmetric Chamfer distance on the xyz channels. a, b: (B, C, N).
    O(N^2) memory per sample - keep N_POINTS modest."""
    pa = a[:, :3].transpose(1, 2)  # (B, N, 3)
    pb = b[:, :3].transpose(1, 2)
    d = torch.cdist(pa, pb)        # (B, N, N)
    return (d.min(dim=2).values.mean(dim=1) + d.min(dim=1).values.mean(dim=1)).mean()


def bound_xyz(out, output_range):
    """Bound the decoder's xyz channels (first 3) to output_range; extra dims
    pass through linearly."""
    if output_range == "neg1_1":
        return torch.cat([torch.tanh(out[:, :3]), out[:, 3:]], dim=1)
    if output_range == "zero_1":
        return torch.cat([torch.sigmoid(out[:, :3]), out[:, 3:]], dim=1)
    return out


class PointNetAE(nn.Module):
    """Bare-bones PointNet autoencoder: shared point MLP -> global max pool ->
    FC encoder to a latent vector -> FC decoder back to (C, N)."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none"):
        super().__init__()
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        self.enc = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(DROPOUT),
        )
        self.enc_fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, n_points * in_channels),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        return self.enc_fc(torch.max(self.enc(x), dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        z = self.encode(x)
        out = self.dec_fc(z).view(-1, self.in_channels, self.n_points)
        return bound_xyz(out, self.output_range), z


# ---- PointNet++ building blocks ---------------------------------------------
def index_points(points, idx):
    """points: (B, N, C); idx: (B, S) or (B, S, G) -> (B, S, C) or (B, S, G, C)."""
    if idx.dim() == 2:
        idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
        return torch.gather(points, 1, idx)
    points = points.unsqueeze(1).expand(-1, idx.shape[1], -1, -1)
    idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
    return torch.gather(points, 2, idx)


def farthest_point_sample(xyz, npoint):
    """Iterative FPS. xyz: (B, N, 3) -> LongTensor (B, npoint).
    Python loop over npoint - fine for a few hundred samples."""
    B, N, _ = xyz.shape
    device = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=device)
    dist = torch.full((B, N), 1e10, device=device)
    farthest = torch.randint(0, N, (B,), device=device)
    batch = torch.arange(B, device=device)
    for i in range(npoint):
        idx[:, i] = farthest
        centroid = xyz[batch, farthest, :].unsqueeze(1)  # (B, 1, 3)
        d = ((xyz - centroid) ** 2).sum(-1)
        dist = torch.min(dist, d)
        farthest = dist.argmax(dim=1)  # (B,) farthest point per batch row
    return idx


def ball_query(radius, nsample, xyz, new_xyz):
    """Ball query: up to nsample points within radius of each centroid;
    centroids with fewer neighbors repeat their nearest point.
    xyz: (B, N, 3), new_xyz: (B, S, 3) -> idx (B, S, nsample)."""
    d = torch.cdist(new_xyz, xyz)                       # (B, S, N)
    order = d.argsort(dim=-1)                           # nearest first
    idx = order[:, :, :nsample]                         # (B, S, nsample)
    in_ball = (torch.gather(d, 2, idx) <= radius)       # (B, S, nsample)
    # replace out-of-ball picks with the nearest point (order[:, :, 0])
    first = order[:, :, :1].expand(-1, -1, nsample)
    return torch.where(in_ball, idx, first)


class SetAbstraction(nn.Module):
    """One PointNet++ set-abstraction level: FPS sample -> ball-query group ->
    shared point MLP on (relative xyz + features) -> max pool per group."""

    def __init__(self, npoint, radius, nsample, in_ch, mlp):
        super().__init__()
        self.npoint, self.radius, self.nsample = npoint, radius, nsample
        layers = []
        ch = in_ch + 3  # relative xyz is concatenated to the point features
        for w in mlp:
            layers += [nn.Conv2d(ch, w, 1), nn.BatchNorm2d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, points):
        # xyz: (B, N, 3); points: (B, C, N)
        new_xyz = index_points(xyz, farthest_point_sample(xyz, self.npoint))  # (B, S, 3)
        idx = ball_query(self.radius, self.nsample, xyz, new_xyz)            # (B, S, G)
        grouped_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)           # (B, S, G, 3)
        grouped_feat = index_points(points.transpose(1, 2), idx)              # (B, S, G, C)
        new_points = torch.cat([grouped_xyz, grouped_feat], dim=-1)
        new_points = self.mlp(new_points.permute(0, 3, 2, 1))                 # (B, W, G, S)
        new_points = torch.max(new_points, dim=2).values                      # (B, W, S)
        return new_xyz, new_points


class FeaturePropagation(nn.Module):
    """One PointNet++ feature-propagation level: k-NN inverse-distance
    interpolation of the coarse features -> concat skip connection -> MLP."""

    def __init__(self, in_ch, mlp):
        super().__init__()
        layers = []
        ch = in_ch
        for w in mlp:
            layers += [nn.Conv1d(ch, w, 1), nn.BatchNorm1d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, new_xyz, points, new_points):
        # xyz: (B, N, 3) target level; new_xyz: (B, M, 3) source level
        # points: (B, C1, N) skip features; new_points: (B, C2, M) coarse features
        M = new_xyz.shape[1]
        if M == 1:
            interp = new_points.expand(-1, -1, xyz.shape[1])
        else:
            k = min(3, M)                                    # 3-NN (fewer if M < 3)
            d = torch.cdist(new_xyz, xyz)                    # (B, M, N)
            dist, idx = d.topk(k, dim=1, largest=False)      # k-NN of each target
            weight = 1.0 / (dist + 1e-8)
            weight = weight / weight.sum(dim=1, keepdim=True)
            interp = (index_points(new_points.transpose(1, 2), idx)
                      * weight.unsqueeze(-1)).sum(dim=1)        # (B, N, C2)
            interp = interp.transpose(1, 2)                  # (B, C2, N)
        new_points = torch.cat([interp, points], dim=1) if points is not None else interp
        return self.mlp(new_points)


class PointNetPP_AE(nn.Module):
    """PointNet++ autoencoder: two set-abstraction levels -> global max pool ->
    FC latent -> FC back to the coarsest level -> feature-propagation decoder
    with skip connections -> per-point head. Decoder xyz output is bounded to
    `output_range`."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none",
                 sa_points=(512, 128), sa_radii=(0.2, 0.4), sa_nsample=32):
        super().__init__()
        if len(sa_points) != len(sa_radii):
            raise ValueError("sa_points and sa_radii must have the same length")
        if max(sa_points) > n_points:
            raise ValueError(f"max(SA_POINTS)={max(sa_points)} exceeds "
                             f"N_POINTS={n_points}")
        if output_range not in ("none", "neg1_1", "zero_1"):
            raise ValueError(f"output_range must be none|neg1_1|zero_1, got {output_range!r}")
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        c0, c1 = 128, 256
        self.sa1 = SetAbstraction(sa_points[0], sa_radii[0], sa_nsample,
                                  in_channels, [64, 64, c0])
        self.sa2 = SetAbstraction(sa_points[1], sa_radii[1], sa_nsample,
                                  c0, [128, 128, c1])
        self.enc_fc = nn.Sequential(
            nn.Linear(c1, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, c1 * sa_points[1]),
        )
        self.fp1 = FeaturePropagation(c1 + c0, [256, 256])
        self.fp2 = FeaturePropagation(256 + in_channels, [256, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Conv1d(128, in_channels, 1),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        return self.enc_fc(torch.max(l2_points, dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        z = self.enc_fc(torch.max(l2_points, dim=2).values)          # (B, L)
        coarse = self.dec_fc(z).view(-1, l2_points.shape[1], l2_xyz.shape[1])
        p1 = self.fp1(l1_xyz, l2_xyz, l1_points, coarse)             # (B, 256, S1)
        p0 = self.fp2(xyz, l1_xyz, x, p1)                            # (B, 128, N)
        return bound_xyz(self.head(p0), self.output_range), z


def build_model(arch, in_channels, n_points, latent_dim, output_range,
                sa_points, sa_radii, sa_nsample):
    if arch == "pointnet":
        return PointNetAE(in_channels, n_points, latent_dim, output_range)
    if arch in ("pointnet++", "pointnetpp"):
        return PointNetPP_AE(in_channels, n_points, latent_dim, output_range,
                             tuple(sa_points), tuple(sa_radii), sa_nsample)
    raise ValueError(f"unknown ARCH {arch!r} (pointnet | pointnet++)")


def make_criterion(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "l1":
        return nn.L1Loss()
    if name == "huber":
        return nn.HuberLoss()
    if name == "chamfer":
        return chamfer_loss
    raise ValueError(f"unknown LOSS {name!r} (mse | l1 | huber | chamfer)")



# ---- load the frozen encoder from the autoencoder checkpoint ---------------
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model = build_model(ckpt["arch"], in_channels=ckpt["in_channels"],
                    n_points=ckpt["n_points"], latent_dim=ckpt["latent_dim"],
                    output_range=ckpt["output_range"], sa_points=ckpt["sa_points"],
                    sa_radii=ckpt["sa_radii"], sa_nsample=ckpt["sa_nsample"]).to(device)
model.load_state_dict(ckpt["state_dict"])
for p in model.parameters():
    p.requires_grad_(False)  # frozen encoder - only the head trains
model.eval()
LATENT_DIM = ckpt["latent_dim"]
print(f"loaded frozen encoder from {CKPT_PATH} "
      f"(arch={ckpt['arch']}, latent_dim={LATENT_DIM})")

# small MLP head on the latent vector
head = nn.Sequential(
    nn.Linear(LATENT_DIM, 256), nn.ReLU(), nn.Dropout(DROPOUT),
    nn.Linear(256, 1),
).to(device)
print(head)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)


def embed(ds):
    # chunked: one forward over ALL plots would build an (n, 1024, N_POINTS)
    # intermediate - tens of GB on a GPU (CUDA OOM)
    Zs = []
    with torch.no_grad():
        for off in range(0, len(ds), EMBED_BATCH):
            chunk = torch.stack([ds[i][0] for i in
                                 range(off, min(off + EMBED_BATCH, len(ds)))])
            Zs.append(model.encode(chunk.to(device)).cpu())
    return torch.cat(Zs)


Z = embed(dataset)                                  # (n_train, LATENT_DIM)
ys_t = torch.tensor(dataset.labels, dtype=torch.float32)
print(f"train embeddings: {tuple(Z.shape)}, finite={bool(torch.isfinite(Z).all())}")

if VAL_FEATURES_DIR is not None:
    # dedicated validation set: all of VAL_FEATURES_DIR/VAL_LABELS_DIR is val
    val_dataset = PlotDataset(VAL_FEATURES_DIR, VAL_LABELS_DIR, LABEL_KEY,
                              EXTRA_DIMS, N_POINTS)
    Z_val = embed(val_dataset)
    ys_val = torch.tensor(val_dataset.labels, dtype=torch.float32)
    print(f"val embeddings: {tuple(Z_val.shape)} (dedicated {VAL_FEATURES_DIR!r})")
    train_idx = torch.arange(len(dataset))
    val_idx = None  # dedicated val set: Z_val/ys_val are used instead
else:
    Z_val = ys_val = None
    n_val = max(1, int(round(len(dataset) * VAL_FRAC))) if len(dataset) > 1 else 0
    perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))
    val_idx, train_idx = perm[:n_val], perm[n_val:]

# z-score the label using train-set stats only
y_mean = ys_t[train_idx].mean()
y_std = ys_t[train_idx].std()
if STANDARDIZE_LABELS and float(y_std) > 0:
    y_fit = (ys_t - y_mean) / y_std
    y_val_fit = (ys_val - y_mean) / y_std if ys_val is not None else None
else:
    y_mean, y_std = torch.tensor(0.0), torch.tensor(1.0)
    y_fit = ys_t.clone()
    y_val_fit = ys_val.clone() if ys_val is not None else None
print(f"label standardization (train): mean={float(y_mean):.6g} std={float(y_std):.6g}")

train_loader = DataLoader(TensorDataset(Z[train_idx], y_fit[train_idx]),
                          batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=len(train_idx) > BATCH_SIZE)
if Z_val is not None:
    val_loader = DataLoader(TensorDataset(Z_val, y_val_fit), batch_size=BATCH_SIZE)
elif val_idx is not None and len(val_idx) > 0:
    val_loader = DataLoader(TensorDataset(Z[val_idx], y_fit[val_idx]),
                            batch_size=BATCH_SIZE)
else:
    val_loader = None
n_val = len(val_loader.dataset) if val_loader is not None else 0

criterion = nn.MSELoss()
opt = torch.optim.Adam(head.parameters(), lr=LR)

best_val = float("inf")
bad = 0
best_state = None
for epoch in range(1, EPOCHS + 1):
    head.train()
    total = 0.0
    n = 0
    for zb, yb in train_loader:
        zb, yb = zb.to(device), yb.to(device)[:, None]
        if zb.shape[0] == 1:
            # single-sample mode: repeat so the batch still has >1 row
            zb = zb.repeat(BATCH_SIZE, 1)
            yb = yb.repeat(BATCH_SIZE, 1)
        opt.zero_grad()
        loss = criterion(head(zb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * zb.shape[0]
        n += zb.shape[0]
    msg = f"epoch {epoch:3d}/{EPOCHS}  train MSE {total / max(n, 1):.6f}"
    if val_loader is not None:
        head.eval()
        vt = 0.0
        vn = 0
        with torch.no_grad():
            for zb, yb in val_loader:
                zb, yb = zb.to(device), yb.to(device)[:, None]
                vt += criterion(head(zb), yb).item() * zb.shape[0]
                vn += zb.shape[0]
        val_loss = vt / max(vn, 1)
        msg += f"  val MSE {val_loss:.6f}"
        if PATIENCE is not None:
            if val_loss < best_val - MIN_DELTA:
                best_val = val_loss
                bad = 0
                best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            else:
                bad += 1
                if bad > PATIENCE:
                    print(f"early stopping at epoch {epoch} "
                          f"(val MSE no improvement for {PATIENCE} epochs)")
                    break
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS or "early stopping" in msg:
        print(msg)

if best_state is not None:
    head.load_state_dict(best_state)
    print(f"restored best head (val MSE {best_val:.6f})")
elif PATIENCE is not None:
    print("PATIENCE set but no validation split - early stopping inactive")


In [ ]:
# ---- evaluate on train AND validation splits, in original units -----------
head.eval()


def eval_split(Zx, yx, name):
    with torch.no_grad():
        pred = head(Zx.to(device)).cpu().squeeze(-1) * y_std + y_mean
    true = yx
    rmse = torch.sqrt(torch.mean((pred - true) ** 2)).item()
    mae = torch.mean(torch.abs(pred - true)).item()
    print(f"{name}: RMSE {rmse:.6g}  MAE {mae:.6g}  ({LABEL_KEY}, n={len(yx)})")
    return true, pred


true_tr, pred_tr = eval_split(Z[train_idx], ys_t[train_idx], "train")
if Z_val is not None:
    true_va, pred_va = eval_split(Z_val, ys_val, "val")
elif n_val:
    true_va, pred_va = eval_split(Z[val_idx], ys_t[val_idx], "val")
else:
    true_va = pred_va = None
    print("no validation split")

try:
    import matplotlib.pyplot as plt
    panels = [(true_tr, pred_tr, "train")]
    if true_va is not None:
        panels.append((true_va, pred_va, "val"))
    fig, axes = plt.subplots(1, len(panels) + 1, figsize=(5 * (len(panels) + 1), 4))
    for ax, (t, p, name) in zip(axes, panels):
        ax.scatter(t.numpy(), p.numpy(), s=20)
        lo = float(min(t.min(), p.min()))
        hi = float(max(t.max(), p.max()))
        ax.plot([lo, hi], [lo, hi], "k--")
        ax.set_xlabel(f"true {LABEL_KEY}")
        ax.set_ylabel("predicted")
        ax.set_title(name)
    axes[-1].hist(Z.norm(dim=1).numpy(), bins=20)
    axes[-1].set_xlabel("embedding norm ||z||")
    axes[-1].set_ylabel("count")
    axes[-1].set_title("embedding norms")
    fig.suptitle(f"label prediction from frozen embeddings ({LABEL_KEY})")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping plots")
